# 03.01 · Memory concepts: short-term vs long-term

**Module 03 · Memory & Context Engineering** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Module 02 (`create_agent`, `ToolRuntime`). |
| 🔑 Keys | `GROQ_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** why a model (and an agent without a checkpointer) forgets everything between calls.
2. **Tell apart** thread-scoped memory (state + checkpointer) and cross-thread memory (store).
3. **Classify** memories as semantic, episodic or procedural, and choose *profile* vs *collection*.
4. **Decide** when to write memories: in the hot path or in the background.
5. **Explain** why `ConversationBufferMemory` and friends are gone, and what replaces them.
6. **Jupyter skill:** develop `src/mastery` live with `%load_ext autoreload` + `%autoreload 2`.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Memory overview](https://docs.langchain.com/oss/python/concepts/memory) | **Main page.** Short- vs long-term, semantic/episodic/procedural, hot path vs background |
| [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence) | Checkpointers, threads and stores in one picture |
| [Context overview](https://docs.langchain.com/oss/python/concepts/context) | Runtime context vs state vs store (used again in 03.05) |
| [LangChain v1 release notes → `langchain-classic`](https://docs.langchain.com/oss/python/releases/langchain-v1#langchain-classic) | Where the legacy chains and memory classes went |
| [IPython autoreload](https://ipython.readthedocs.io/en/stable/config/extensions/autoreload.html) | The Jupyter skill of this module (modes and caveats) |

---
## Step 0 · Install and set up

Module 03 adds one package, `langgraph-checkpoint-sqlite` (SQLite checkpointer and store). If you ran `python -m pip install -e ".[dev]"` after pulling Module 03, you already have it.

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2" "langgraph-checkpoint-sqlite>=3.1.1,<4"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langgraph", "langgraph-checkpoint", "langgraph-checkpoint-sqlite",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/03-memory-and-context/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Setup 4: Silence one known, harmless warning, only on the langchain-core versions that raise it.
# Older langchain-core 1.6.x serialized the injected ToolRuntime while parsing tool input, and Pydantic
# warned "PydanticSerializationUnexpectedValue ... field_name='context'" (context still arrived intact).
# Current releases don't: on them this cell changes nothing.
import warnings
from importlib.metadata import version

if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

---
## Part 1 · Jupyter skill: edit `src/mastery` without restarting the kernel

Until now, after editing a file in `src/mastery/` you restarted the kernel (Lesson 00.05). In this module you'll grow `mastery/memory.py`, and restarting every time loses your in-memory checkpointer and store. IPython's **autoreload** extension re-imports changed modules **before each cell runs**.

| Command | Effect |
|---|---|
| `%load_ext autoreload` | Load the extension (once per kernel) |
| `%autoreload 2` (or `all`) | Reload every changed module before each cell. **Use this one.** |
| `%autoreload 1` (or `explicit`) + `%aimport mastery.memory` | Reload only modules you list |
| `%autoreload 3` (or `complete`) | Like `2`, and also picks up **new** names added to a module |
| `%autoreload 0` (or `off`) | Stop |
| `%autoreload 2 -p` | Same as `2`, printing what was reloaded |

We'll prove it works with a throw-away module in `data/outputs/scratch/` (git-ignored), so nothing in `src/` changes.

In [ ]:
# 🧪 Offline: turn autoreload on. Put these two lines at the top of notebooks where you edit src/mastery.
%load_ext autoreload
%autoreload 2

In [ ]:
# 🧪 Offline: write a tiny module and import it.
import importlib
import time

SCRATCH = REPO_ROOT / "data" / "outputs" / "scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)
module_file = SCRATCH / "greeter_demo.py"
module_file.write_text('def greet(name):\n    return f"Hello, {name}!"\n', encoding="utf-8")
if str(SCRATCH) not in sys.path:
    sys.path.insert(0, str(SCRATCH))
importlib.invalidate_caches()

import greeter_demo
from greeter_demo import greet

print("before edit:", greet("Ana"))

In [ ]:
# 🧪 Offline: now change the file on disk, as if you had edited it in VS Code.
# (In a separate cell: autoreload notes each module's file time before a cell runs.)
time.sleep(1.1)  # file times can have 1-second resolution; make sure the edit is "newer".
_ = module_file.write_text('def greet(name):\n    return f"¡Hola, {name}! (edited on disk)"\n', encoding="utf-8")
print("edited", module_file.name)

In [ ]:
# 🧪 Offline: a NEW cell. autoreload runs first, so the edited code is used, even via `from ... import`.
print("after edit :", greet("Ana"))

**You should now see** `before edit: Hello, Ana!`, `edited greeter_demo.py`, and then `after edit : ¡Hola, Ana! (edited on disk)`. You didn't re-import anything: autoreload swapped the function's code before the second cell ran.

### What just happened?

- Before each cell, autoreload checks the modification time of every imported module's file. Changed files are re-imported, and existing functions and classes are **patched in place**, so names you got with `from greeter_demo import greet` see the new code too.
- The check happens **before** each cell. A module seen for the first time is only *recorded*; it's reloaded when its file is newer than the recorded time. That's why import, edit and call are three cells.

**Caveats (from the IPython docs):** C extensions can't be reloaded; turning a method into a property (or the reverse) can confuse it; objects you already built (an agent, a checkpointer) keep the data they were built with, so **re-run the cell that builds them** after changing a factory. When something looks odd, restart the kernel: that is always correct.

> **Windows note:** editors like VS Code save files with a fresh modification time, so autoreload notices. If you edit through a OneDrive/Dropbox-synced folder and a change is not picked up, save again or run `%autoreload now`.

---
## Part 2 · Models are stateless: "memory" is what you send back

A chat model call is a pure function of the messages you send. If the second call doesn't include the first exchange, the model can't know it happened. Agents are the same: without a **checkpointer**, every `invoke` starts from an empty state.

In [ ]:
# 🧪 Offline: the scripted model reports how many messages it received on each call.
from langchain.agents import create_agent
from mastery.testing import ScriptedChatModel


def counting_model() -> ScriptedChatModel:
    return ScriptedChatModel(responses=[lambda msgs: f"(I received {len(msgs)} message(s).)"], repeat_last=True)


forgetful = create_agent(counting_model(), tools=[])
for text in ("Hi, I'm Ana and I'm learning fractions.", "What am I learning?"):
    out = forgetful.invoke({"messages": [{"role": "user", "content": text}]})
    print(f"{text:<42} → {out['messages'][-1].content}")

**You should now see** `(I received 1 message(s).)` twice. The second call never saw *"I'm learning fractions"*.

In [ ]:
# 💸 Live: a real model can't answer either (1 short call per question).
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    live_forgetful = create_agent(get_model("fast", temperature=0, reasoning_effort="low"), tools=[])
    live_forgetful.invoke({"messages": [{"role": "user", "content": "Hi, I'm Ana and I'm learning fractions."}]})
    reply = live_forgetful.invoke({"messages": [{"role": "user", "content": "What am I learning? One sentence."}]})
    print(reply["messages"][-1].content)
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** something like *"I don't know what you're learning, could you tell me?"*. Not a bug: nothing told the model.

---
## Part 3 · Two scopes: thread memory and cross-thread memory

| | **Short-term** (thread-scoped) | **Long-term** (cross-thread) |
|---|---|---|
| What | The agent's **state**: messages + your fields | Facts, preferences, past experiences |
| Lives in | A **checkpointer** (`InMemorySaver`, `SqliteSaver`, `PostgresSaver`) | A **store** (`InMemoryStore`, `SqliteStore`, `PostgresStore`) |
| Key | `thread_id` in `config["configurable"]` | A **namespace** tuple + key, e.g. `("learners", "ana")` / `"profile"` |
| Written | Automatically, after every step | **By your code**: tools, middleware, background jobs |
| Read | Automatically, at the start of every step | By your code: `runtime.store.get/search` |
| Like… | One email thread | Your contact card for that person |
| Lesson | 03.02, 03.03 | 03.04 |

`mastery.memory` (new in this module) builds both with one line each. In-memory versions for tests, SQLite for notebooks.

In [ ]:
# 🧪 Offline: the same agent with a checkpointer remembers within a thread, not across threads.
from mastery.memory import learner_namespace, make_checkpointer, make_store, thread_config

remembering = create_agent(counting_model(), tools=[], checkpointer=make_checkpointer("memory"))
for thread, text in [("t-1", "Hi, I'm Ana."), ("t-1", "What's my name?"), ("t-2", "What's my name?")]:
    out = remembering.invoke({"messages": [{"role": "user", "content": text}]}, thread_config(thread))
    print(f"thread {thread}: {text:<16} → {out['messages'][-1].content}")

# The store is a separate, explicit key-value space that every thread can read.
store = make_store("memory")
store.put(learner_namespace("ana"), "profile", {"name": "Ana", "learning": "fractions"})
print("\nstore:", store.get(learner_namespace("ana"), "profile").value)

**You should now see** `thread t-1 ... (I received 1 message(s).)`, then `(I received 3 message(s).)` (the first question, the first answer, the new question), and `thread t-2 ... (I received 1 message(s).)`. The last line prints Ana's profile from the store.

### What just happened?

- With `checkpointer=...`, `create_agent` **loads** the saved state of the thread before running and **saves** it after each step. The model received the earlier messages because the graph put them back into the state.
- A new `thread_id` is a new, empty conversation. Nothing leaks between threads.
- The store didn't do anything automatically. Long-term memory is **always explicit**: someone decides what to write and when to read it. That's the subject of Lesson 03.04.

---
## Part 4 · What kind of memory? Semantic, episodic, procedural

The memory overview borrows three categories from psychology:

| Type | Stores | Study-buddy example | Typical shape |
|---|---|---|---|
| **Semantic** | Facts | "Ana is 14", "weak topic: fractions" | A **profile** (one JSON document you update) or a **collection** (many small documents you add to) |
| **Episodic** | Experiences | "On 2026-10-05 the pizza analogy worked for fractions" | Past runs, or few-shot examples of good answers |
| **Procedural** | How to behave | "Explain with examples first, then the rule" | Instructions, often part of the system prompt, sometimes updated by the agent |

**Profile vs collection.** A profile is easy to read and to inject (one `get`), but every update rewrites it and the model can drop fields. A collection is append-only and easy to write, but you need search (Lesson 03.04) to find the right items. Many apps use both: a small profile + a searchable collection.

**When to write memories:**

| | In the hot path | In the background |
|---|---|---|
| How | A tool the agent calls (`remember_fact`) during the conversation | A separate job after (or between) conversations |
| Pro | Immediately available; the user can see it happen | No extra latency; can look at the whole conversation |
| Con | Slower replies; the agent must decide *while* answering | Memories appear later; needs a scheduler |
| In this course | Lesson 03.04 and Project 03 | Lesson 11.07 (background consolidation with Deep Agents) |

In [ ]:
# 🧪 Offline: a profile (one document, updated) next to a collection (many documents, appended).
import uuid

ns = learner_namespace("ana")
profile = store.get(ns, "profile").value
store.put(ns, "profile", {**profile, "style": "examples first"})      # update = overwrite the whole value

episodes = learner_namespace("ana", "episodes")
for note in ("pizza slices made 3/8 click", "number line did not help with 1/2 + 1/3"):
    store.put(episodes, str(uuid.uuid4()), {"note": note})           # append = new key each time

print("profile   :", store.get(ns, "profile").value)
print("episodes  :", [item.value["note"] for item in store.search(episodes)])
print("namespaces:", store.list_namespaces(prefix=("learners",)))

**You should now see** the profile with a new `style` field, the two episode notes, and two namespaces: `('learners', 'ana')` and `('learners', 'ana', 'episodes')`.

---
## Part 5 · Why `ConversationBufferMemory` is gone

Before LangChain 1.0, "memory" meant classes like `ConversationBufferMemory`, `ConversationBufferWindowMemory` and `ConversationSummaryMemory`, attached to a chain. They kept the history **inside a Python object**, so it vanished on restart, couldn't be shared by two server processes, and knew nothing about tools or threads.

In v1 they live in the separate `langchain-classic` package, deprecated, with this note in their source: *"use `create_agent` with checkpointing or the `Store` API"*. The replacements:

| Old (pre-1.0) | New (v1) | Lesson |
|---|---|---|
| `ConversationBufferMemory` | `checkpointer=` + `thread_id` | 03.02 |
| `ConversationBufferWindowMemory(k=...)` | Trim in `@before_model` middleware / `trim_messages` | 03.03 |
| `ConversationSummaryMemory` | `SummarizationMiddleware` | 03.03 |
| `VectorStoreRetrieverMemory` | Store with an `index` + `store.search(query=...)` | 03.04 |
| `memory_key="chat_history"` prompt variables | Messages in state; dynamic prompts from middleware | 03.05 |

In [ ]:
# 🧪 Offline: the old import path doesn't exist in langchain 1.x.
import importlib.util

print("langchain.memory importable:", importlib.util.find_spec("langchain.memory") is not None)

**You should now see** `langchain.memory importable: False`. Old tutorials that start with `from langchain.memory import ...` are pre-1.0. Translate them with the table above instead of installing `langchain-classic`.

---
## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| The agent forgets the previous message | No `checkpointer=` or no `thread_id` in the config | Pass both (Part 3, Lesson 03.02) |
| `ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id ...` | Checkpointer given but `invoke` had no config | `agent.invoke(input, thread_config("..."))` |
| Two users see each other's history | Both used the same `thread_id` | Build thread ids from the user (and session) id on the server |
| `ModuleNotFoundError: langchain.memory` | Pre-1.0 tutorial | Use the replacement table in Part 5 |
| `%autoreload` doesn't pick up a change | Edit and call in the same cell, or the object was built before the change | Call in a new cell; re-run the cell that builds the object; restart if in doubt |
| `UsageError: Line magic function %autoreload not found` | `%load_ext autoreload` not run in this kernel | Run it first (once per kernel) |
| `InvalidNamespaceError: ... cannot contain periods` | A namespace label like `"ana.perez"` | Labels can't contain `.` or be empty; `mastery.memory.safe_label` checks this |

## ⚠️ Production notes

> ⚠️ **Production note: memory is personal data.**
> Long-term memory stores facts about people. Decide up front what you may store, for how long (store TTLs, Lesson 03.04), and how a user can see and delete it. Under data-protection laws (GDPR; in Colombia, Ley 1581 de 2012), "forget me" must actually delete from the store **and** the checkpointer (`checkpointer.delete_thread(...)`).

> ⚠️ **Production note: thread ids are a security boundary.**
> Whoever knows a `thread_id` can resume that conversation. Generate them on the server (e.g. `uuid4`), store them next to the user who owns them, and never accept a raw thread id from the client without checking ownership.

> ⚠️ **Production note: in-memory savers are for tests.**
> `InMemorySaver` and `InMemoryStore` lose everything on restart and aren't shared between processes. Use SQLite for a single-process app or notebook, Postgres for a server (Module 12).

## 🏋️ Exercises

1. **(Easy)** Run Part 3 again with a third question in thread `t-1`. Predict the message count before you run it.
2. **(Easy)** Classify each as semantic, episodic or procedural: *"prefers Spanish"*, *"failed the quiz on 2026-10-01"*, *"always ask one check question at the end"*, *"is preparing for the ICFES exam"*.
3. **(Medium)** Add a function `describe(store, user_id)` that returns the profile plus the number of episodes. Put it in `data/outputs/scratch/greeter_demo.py`, call it, then change its output format **without restarting** (autoreload).
4. **(Stretch)** Look at `store.search(episodes, limit=1)` and `offset=1`. How would you page through 1,000 episodes?

<details>
<summary><b>Show solutions</b></summary>

**1.** 5: question 1, answer 1, question 2, answer 2, question 3.

**2.** Semantic, episodic, procedural, semantic.

**4.** Loop with `offset += limit` until a page comes back empty (or shorter than `limit`). The stores documentation shows the same loop with `if not page: break`, and warns that ordering differs by backend.
</details>

## ✅ Checkpoint

- [ ] I can explain why an agent without a checkpointer forgets.
- [ ] I can say where short-term and long-term memory live, and how each is keyed.
- [ ] I can give a semantic, an episodic and a procedural memory for my own project idea.
- [ ] I can explain profile vs collection, and hot path vs background writes.
- [ ] I know what replaces `ConversationBufferMemory` in v1.
- [ ] I can use `%load_ext autoreload` + `%autoreload 2` and know its caveats.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Docs drill (5 min)

`ConversationBufferMemory` still shows up in old tutorials. Find its deprecation notice: since which version, removed when, and what does it recommend instead?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Reference: search `ConversationBufferMemory` (langchain-classic) on reference.langchain.com; or GitHub `libs/langchain/langchain_classic/memory/buffer.py`, the `@deprecated` decorator. (`langchain-classic` isn't installed in the course env, so use the page or GitHub.)
2. Answer: deprecated since 0.3.1, to be removed in 2.0.0; use `create_agent` with checkpointing, or the Store API.
3. Then check langchain-classic's PyPI release history: is it still released? Log the date.

</details>

## 🔎 Research checkpoint

Open the [feature comparison table](https://docs.langchain.com/oss/python/concepts/products#feature-comparison) and compare the **Short-term memory** and **Long-term memory** rows for LangGraph, LangChain and Deep Agents (follow each link). For a tutoring app that must remember a learner for **three months**, which would you pick, and why? Log your answer in [`docs/research-log.md`](../../docs/research-log.md). (This is the module's research checkpoint; you'll revisit your answer after 03.04.)

## Summary

- Models are stateless; an agent remembers only what its state gives back to the model.
- Short-term memory = state saved by a checkpointer, per `thread_id`. Long-term memory = a store, per namespace, written explicitly.
- Semantic (facts), episodic (experiences), procedural (instructions). Profiles vs collections. Hot path vs background.
- Legacy memory classes are deprecated in `langchain-classic`; checkpointers, middleware and stores replace them.
- `%autoreload 2` lets you edit `src/mastery` while you work, with known caveats.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 03.02 · Short-term memory with checkpointers and threads** → [`03-02-short-term-memory.ipynb`](03-02-short-term-memory.ipynb).

[Module 03 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)